# 03 — Build circuit features

## Purpose

Merge canonical circuit characteristics onto every 2004–2025 driver-race row. Create centered circuit scores and reproducible primary-trait labels at the unique-circuit level before expanding them to race rows.

The canonical metadata contains one static profile per circuit. Layout changes across seasons are therefore an acknowledged project limitation rather than an unobserved season-specific adjustment.

## Inputs

- `data/processed/driver_race_dataset.csv`
- `data/external/f1_circuit_metadata.csv`

The external metadata file remains the authoritative source for verbose methodology, evidence, and research URLs. Notebook 3 merges analytical fields only to avoid repeating long provenance text across thousands of rows.

## Outputs

- `data/processed/driver_race_with_circuit_features.csv`
- `data/outputs/notebook_03_validation_report.json`

## Imports

In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Dict, Iterable, Mapping, Sequence

import numpy as np
import pandas as pd

## Configuration

In [2]:
START_YEAR = 2004
END_YEAR = 2025

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
EXTERNAL_DIR = PROJECT_ROOT / "data" / "external"
OUTPUT_DIR = PROJECT_ROOT / "data" / "outputs"

DRIVER_RACE_PATH = PROCESSED_DIR / "driver_race_dataset.csv"
METADATA_PATH = EXTERNAL_DIR / "f1_circuit_metadata.csv"
DATASET_PATH = PROCESSED_DIR / "driver_race_with_circuit_features.csv"
VALIDATION_REPORT_PATH = OUTPUT_DIR / "notebook_03_validation_report.json"

DRIVER_RACE_KEY = ["season", "round", "driver_id"]
TRAITS = ("downforce", "speed", "technical", "overtaking")
SCORE_COLUMNS = [f"{trait}_score" for trait in TRAITS]
ZSCORE_COLUMNS = [f"{trait}_zscore" for trait in TRAITS]
CENTERED_COLUMNS = [f"{trait}_centered" for trait in TRAITS]
PERCENTILE_COLUMNS = [f"{trait}_percentile" for trait in TRAITS]

BINARY_COLUMNS = [
    "is_street_circuit", "is_permanent_circuit", "is_hybrid_circuit",
    "high_downforce_estimate", "low_downforce_estimate",
    "high_speed_estimate", "low_speed_estimate",
    "high_technical_estimate", "low_technical_estimate",
    "overtaking_friendly_estimate", "overtaking_difficult_estimate",
    "long_lap_estimate", "short_lap_estimate",
    "high_altitude_estimate", "low_altitude_estimate",
]

ANALYTICAL_METADATA_COLUMNS = [
    "circuit_id", "circuit_name", "circuit_locality", "circuit_country",
    "circuit_type", "lap_length_km", "altitude_m",
    "downforce_score", "speed_score", "technical_score", "overtaking_score",
    "downforce_level", "speed_profile", "technical_level", "overtaking_level",
    "lap_length_bucket", "altitude_bucket",
    "is_street_circuit", "is_permanent_circuit", "is_hybrid_circuit",
    "downforce_zscore", "speed_zscore", "technical_zscore", "overtaking_zscore",
    "downforce_percentile", "speed_percentile",
    "technical_percentile", "overtaking_percentile",
    "high_downforce_estimate", "low_downforce_estimate",
    "high_speed_estimate", "low_speed_estimate",
    "high_technical_estimate", "low_technical_estimate",
    "overtaking_friendly_estimate", "overtaking_difficult_estimate",
    "lap_length_zscore", "altitude_zscore",
    "long_lap_estimate", "short_lap_estimate",
    "high_altitude_estimate", "low_altitude_estimate",
    "driver_precision_index", "power_speed_index",
    "setup_compromise_index", "passing_opportunity_index",
    "track_profile_tags", "score_confidence", "metadata_confidence",
]

VERBOSE_PROVENANCE_COLUMNS = {
    "rating_methodology", "rating_scale_definition", "rating_evidence_summary",
    "lap_length_source", "altitude_source", "classification_source",
    "specific_research_sources", "global_research_sources_used", "notes",
}

for directory in (PROCESSED_DIR, OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

## Helper functions

In [3]:
def require_files(paths: Sequence[Path]) -> None:
    """Raise a readable error when a required input is absent."""
    missing = [str(path) for path in paths if not path.exists()]
    if missing:
        raise FileNotFoundError(f"Missing required inputs: {missing}")


def filter_scope(frame: pd.DataFrame) -> pd.DataFrame:
    """Restrict rows to the configured inclusive season range."""
    seasons = pd.to_numeric(frame["season"], errors="coerce")
    return frame.loc[seasons.between(START_YEAR, END_YEAR)].reset_index(drop=True)


def build_primary_trait(row: pd.Series) -> pd.Series:
    """Return a non-arbitrary primary label and complete tied-trait detail."""
    scores = {trait: row[f"{trait}_score"] for trait in TRAITS}
    maximum = max(scores.values())
    leaders = [trait for trait in TRAITS if scores[trait] == maximum]
    return pd.Series(
        {
            "primary_trait": leaders[0] if len(leaders) == 1 else "mixed",
            "primary_trait_detail": "; ".join(leaders),
            "primary_trait_tie_count": len(leaders),
        }
    )


def save_csv(frame: pd.DataFrame, path: Path) -> None:
    """Write a CSV atomically so a partial file is never treated as final."""
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary_path, index=False)
    temporary_path.replace(path)


def report_check(name: str, passed: bool, details: Any) -> Dict[str, Any]:
    """Create a JSON-serializable validation record."""
    return {"check": name, "passed": bool(passed), "details": details}


def normalized_csv(frame: pd.DataFrame) -> pd.DataFrame:
    """Normalize values through strings for a CSV round-trip comparison."""
    return frame.reset_index(drop=True).astype("string").fillna("").astype(str)

## Processing

In [4]:
require_files([DRIVER_RACE_PATH, METADATA_PATH])

driver_race_dataset = filter_scope(pd.read_csv(DRIVER_RACE_PATH))
canonical_metadata = pd.read_csv(METADATA_PATH)

missing_required_metadata_columns = sorted(
    set(ANALYTICAL_METADATA_COLUMNS) - set(canonical_metadata.columns)
)
if missing_required_metadata_columns:
    raise ValueError(
        f"Canonical metadata is missing required columns: {missing_required_metadata_columns}"
    )

# All transformations occur once per unique circuit, never on repeated driver-race rows.
circuit_features = canonical_metadata[ANALYTICAL_METADATA_COLUMNS].copy()
for trait in TRAITS:
    score_column = f"{trait}_score"
    circuit_features[f"{trait}_centered"] = (
        circuit_features[score_column] - circuit_features[score_column].mean()
    )

primary_traits = circuit_features.apply(build_primary_trait, axis=1)
circuit_features = pd.concat([circuit_features, primary_traits], axis=1)
circuit_features["circuit_metadata_static_flag"] = 1

overlapping_columns = sorted(
    (set(driver_race_dataset.columns) & set(circuit_features.columns)) - {"circuit_id"}
)
if overlapping_columns:
    raise ValueError(
        f"Unexpected column collisions before circuit merge: {overlapping_columns}"
    )

driver_race_with_circuit_features = driver_race_dataset.merge(
    circuit_features,
    on="circuit_id",
    how="left",
    validate="many_to_one",
)
driver_race_with_circuit_features = driver_race_with_circuit_features.sort_values(
    ["season", "round", "position", "driver_id"]
).reset_index(drop=True)

## Diagnostics

In [5]:
used_circuit_ids = set(driver_race_dataset["circuit_id"])
metadata_circuit_ids = set(circuit_features["circuit_id"])
trait_correlations = circuit_features[SCORE_COLUMNS].corr().round(3)

diagnostics = pd.DataFrame(
    {
        "metric": [
            "driver-race rows", "dataset circuits", "metadata circuits",
            "missing metadata circuits", "unused metadata circuits",
            "unique-primary circuits", "mixed-primary circuits",
            "high-confidence circuits", "medium-confidence circuits",
        ],
        "value": [
            len(driver_race_with_circuit_features),
            len(used_circuit_ids),
            len(metadata_circuit_ids),
            len(used_circuit_ids - metadata_circuit_ids),
            len(metadata_circuit_ids - used_circuit_ids),
            int(circuit_features["primary_trait"].ne("mixed").sum()),
            int(circuit_features["primary_trait"].eq("mixed").sum()),
            int(circuit_features["metadata_confidence"].eq("high").sum()),
            int(circuit_features["metadata_confidence"].eq("medium").sum()),
        ],
    }
)
display(diagnostics)
display(trait_correlations)
display(
    circuit_features.loc[
        circuit_features["primary_trait"].eq("mixed"),
        ["circuit_id", *SCORE_COLUMNS, "primary_trait_detail"],
    ]
)
print(
    "Limitation: one static circuit profile is applied across all seasons in which "
    "that circuit appears."
)

,metric,value
0,driver-race rows,9125
1,dataset circuits,38
2,metadata circuits,38
3,missing metadata circuits,0
4,unused metadata circuits,0
5,unique-primary circuits,27
6,mixed-primary circuits,11
7,high-confidence circuits,21
8,medium-confidence circuits,17


,downforce_score,speed_score,technical_score,overtaking_score
downforce_score,1.000,-0.739,0.738,-0.671
speed_score,-0.739,1.000,-0.401,0.642
technical_score,0.738,-0.401,1.000,-0.511
overtaking_score,-0.671,0.642,-0.511,1.000


,circuit_id,downforce_score,speed_score,technical_score,overtaking_score,primary_trait_detail
5,catalunya,8,6,8,4,downforce; technical
7,hockenheimring,5,7,5,7,speed; overtaking
12,istanbul,6,8,8,6,speed; technical
15,magny_cours,7,6,7,6,downforce; technical
16,marina_bay,9,3,9,4,downforce; technical
18,monaco,10,1,10,1,downforce; technical
20,mugello,7,8,8,4,speed; technical
23,red_bull_ring,4,8,5,8,speed; overtaking
25,rodriguez,8,8,6,8,downforce; speed; overtaking
34,villeneuve,3,8,5,8,speed; overtaking


Limitation: one static circuit profile is applied across all seasons in which that circuit appears.


## Save outputs

In [6]:
save_csv(driver_race_with_circuit_features, DATASET_PATH)
print(
    f"Saved {len(driver_race_with_circuit_features):,} rows to "
    f"{DATASET_PATH.relative_to(PROJECT_ROOT)}"
)

Saved 9,125 rows to data/processed/driver_race_with_circuit_features.csv


## Validation

In [7]:
used_circuit_ids = set(driver_race_dataset["circuit_id"])
metadata_circuit_ids = set(circuit_features["circuit_id"])
output_keys = set(
    map(tuple, driver_race_with_circuit_features[DRIVER_RACE_KEY].to_numpy())
)
input_keys = set(map(tuple, driver_race_dataset[DRIVER_RACE_KEY].to_numpy()))

score_range_issues = {
    column: int((~circuit_features[column].between(1, 10)).sum())
    for column in SCORE_COLUMNS
}
centered_means = circuit_features[CENTERED_COLUMNS].mean().to_dict()
zscore_differences = {}
for trait in TRAITS:
    score = circuit_features[f"{trait}_score"]
    recomputed = (score - score.mean()) / score.std(ddof=0)
    difference = np.max(
        np.abs(recomputed.to_numpy() - circuit_features[f"{trait}_zscore"].to_numpy())
    )
    zscore_differences[trait] = float(difference)

binary_values = {
    column: sorted(circuit_features[column].dropna().unique().tolist())
    for column in BINARY_COLUMNS
}
circuit_type_flag_sum = circuit_features[
    ["is_street_circuit", "is_permanent_circuit", "is_hybrid_circuit"]
].sum(axis=1)

recomputed_primary_traits = circuit_features.apply(build_primary_trait, axis=1)
primary_trait_equal = recomputed_primary_traits.equals(
    circuit_features[
        ["primary_trait", "primary_trait_detail", "primary_trait_tie_count"]
    ]
)

feature_columns = [
    column for column in circuit_features.columns if column != "circuit_id"
]
missing_feature_values = (
    driver_race_with_circuit_features[feature_columns].isna().sum()
)
missing_feature_values = missing_feature_values[
    missing_feature_values.gt(0)
].to_dict()

expected_output_columns = {
    "circuit_name", "circuit_type", "primary_trait", "primary_trait_detail",
    "downforce_score", "speed_score", "technical_score", "overtaking_score",
    "downforce_centered", "speed_centered",
    "technical_centered", "overtaking_centered",
    "downforce_zscore", "speed_zscore",
    "technical_zscore", "overtaking_zscore",
    "circuit_metadata_static_flag", "metadata_confidence",
}

checks = [
    report_check(
        "one output row per driver-race input row",
        len(driver_race_with_circuit_features) == len(driver_race_dataset)
        and output_keys == input_keys,
        {
            "input_rows": len(driver_race_dataset),
            "output_rows": len(driver_race_with_circuit_features),
        },
    ),
    report_check(
        "unique driver-race keys preserved",
        not driver_race_with_circuit_features.duplicated(DRIVER_RACE_KEY).any(),
        int(driver_race_with_circuit_features.duplicated(DRIVER_RACE_KEY).sum()),
    ),
    report_check(
        "metadata circuit IDs are unique",
        not circuit_features["circuit_id"].duplicated().any(),
        int(circuit_features["circuit_id"].duplicated().sum()),
    ),
    report_check(
        "complete and exact circuit coverage",
        used_circuit_ids == metadata_circuit_ids,
        {
            "missing": sorted(used_circuit_ids - metadata_circuit_ids),
            "unused": sorted(metadata_circuit_ids - used_circuit_ids),
        },
    ),
    report_check(
        "no missing merged analytical features",
        len(missing_feature_values) == 0,
        missing_feature_values,
    ),
    report_check(
        "raw circuit scores are within 1 to 10",
        all(count == 0 for count in score_range_issues.values()),
        score_range_issues,
    ),
    report_check(
        "centered scores average zero across unique circuits",
        all(abs(value) < 1e-12 for value in centered_means.values()),
        centered_means,
    ),
    report_check(
        "stored z-scores match population z-scores",
        all(value < 1e-12 for value in zscore_differences.values()),
        zscore_differences,
    ),
    report_check(
        "percentiles are within zero and one",
        all(
            circuit_features[column].between(0, 1).all()
            for column in PERCENTILE_COLUMNS
        ),
        {
            column: [
                float(circuit_features[column].min()),
                float(circuit_features[column].max()),
            ]
            for column in PERCENTILE_COLUMNS
        },
    ),
    report_check(
        "binary flags contain only zero and one",
        all(set(values).issubset({0, 1}) for values in binary_values.values()),
        binary_values,
    ),
    report_check(
        "circuit-type flags are mutually exclusive and exhaustive",
        circuit_type_flag_sum.eq(1).all(),
        circuit_type_flag_sum.value_counts().sort_index().to_dict(),
    ),
    report_check(
        "primary-trait labels are reproducible",
        primary_trait_equal,
        {
            "unique": int(circuit_features["primary_trait"].ne("mixed").sum()),
            "mixed": int(circuit_features["primary_trait"].eq("mixed").sum()),
        },
    ),
    report_check(
        "static metadata limitation is explicit on every row",
        driver_race_with_circuit_features[
            "circuit_metadata_static_flag"
        ].eq(1).all(),
        {"static_rows": int(
            driver_race_with_circuit_features[
                "circuit_metadata_static_flag"
            ].sum()
        )},
    ),
    report_check(
        "verbose provenance remains in canonical metadata only",
        not bool(
            VERBOSE_PROVENANCE_COLUMNS
            & set(driver_race_with_circuit_features.columns)
        ),
        sorted(
            VERBOSE_PROVENANCE_COLUMNS
            & set(driver_race_with_circuit_features.columns)
        ),
    ),
    report_check(
        "expected output columns",
        expected_output_columns
        <= set(driver_race_with_circuit_features.columns),
        sorted(
            expected_output_columns
            - set(driver_race_with_circuit_features.columns)
        ),
    ),
]

validation_report = {
    "notebook": "03_build_circuit_features",
    "scope": {"start_year": START_YEAR, "end_year": END_YEAR},
    "status": "PASS" if all(check["passed"] for check in checks) else "FAIL",
    "checks": checks,
}
display(pd.DataFrame(checks))
if validation_report["status"] != "PASS":
    failed = [check for check in checks if not check["passed"]]
    raise AssertionError(f"Validation failed: {failed}")
print("PASS")

,check,passed,details
0,one output row per driver-race input row,True,"{'input_rows': 9125, 'output_rows': 9125}"
1,unique driver-race keys preserved,True,0
2,metadata circuit IDs are unique,True,0
3,complete and exact circuit coverage,True,"{'missing': [], 'unused': []}"
4,no missing merged analytical features,True,{}
5,raw circuit scores are within 1 to 10,True,"{'downforce_score': 0, 'speed_score': 0, 'tech..."
6,centered scores average zero across unique cir...,True,"{'downforce_centered': -9.349246523159212e-17,..."
7,stored z-scores match population z-scores,True,"{'downforce': 4.440892098500626e-16, 'speed': ..."
8,percentiles are within zero and one,True,"{'downforce_percentile': [0.0263157894736842, ..."
9,binary flags contain only zero and one,True,"{'is_street_circuit': [0, 1], 'is_permanent_ci..."


PASS


## Supercheck

In [8]:
saved_dataset = pd.read_csv(DATASET_PATH, dtype=str, keep_default_na=False)
dataset_equal = normalized_csv(driver_race_with_circuit_features).equals(
    normalized_csv(saved_dataset)
)
superchecks = [
    report_check(
        "driver-race circuit-feature file exists",
        DATASET_PATH.exists(),
        str(DATASET_PATH),
    ),
    report_check(
        "saved dataset equals in-memory dataset",
        dataset_equal,
        {"rows": len(saved_dataset), "columns": len(saved_dataset.columns)},
    ),
]
validation_report["superchecks"] = superchecks
validation_report["status"] = (
    "PASS" if all(check["passed"] for check in checks + superchecks) else "FAIL"
)

serializable_report = json.loads(json.dumps(validation_report, default=str))
temporary_report_path = VALIDATION_REPORT_PATH.with_suffix(".json.tmp")
with temporary_report_path.open("w", encoding="utf-8") as handle:
    json.dump(serializable_report, handle, indent=2)
temporary_report_path.replace(VALIDATION_REPORT_PATH)

saved_report = json.loads(VALIDATION_REPORT_PATH.read_text(encoding="utf-8"))
report_checks = [
    report_check(
        "validation report file exists",
        VALIDATION_REPORT_PATH.exists(),
        str(VALIDATION_REPORT_PATH),
    ),
    report_check(
        "saved validation report equals memory",
        saved_report == serializable_report,
        None,
    ),
]
display(pd.DataFrame(superchecks + report_checks))
if validation_report["status"] != "PASS" or not all(
    check["passed"] for check in report_checks
):
    raise AssertionError("Supercheck failed")
print("PASS")

,check,passed,details
0,driver-race circuit-feature file exists,True,
1,saved dataset equals in-memory dataset,True,"{'rows': 9125, 'columns': 117}"
2,validation report file exists,True,
3,saved validation report equals memory,True,None


PASS
